In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.reorg


>DOM reorganization
output-file: core.dom.reorg
title: core.dom.reorg

This notebook provides utility functions for DOM reorganization
---

In [ ]:
# | default_exp core.dom.reorg

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
import json
import copy
import os
import re
from collections.abc import Callable
from typing import Optional

from collections.abc import Iterator
import pypandoc

In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | hide
from unittest.mock import patch

def _header(level: int, text: str) -> dict:
    return {
        't': 'Header',
        'c': [
            level,
            [f'h{level}-{text}', [], []],
            [{'t': 'Str', 'c': text}],
        ],
    }


def _para(text: str) -> dict:
    return {'t': 'Para', 'c': [{'t': 'Str', 'c': text}]}

In [ ]:
# | export
_TOC_ENTRY_RE = re.compile(r"^.+\s+\d+$")


def _node_text(node) -> str:
    """Extract readable text from a small Pandoc AST fragment."""
    if isinstance(node, dict):
        node_type = node.get('t')
        if node_type == 'Str':
            return str(node.get('c', ''))
        if node_type in {'Space', 'SoftBreak', 'LineBreak'}:
            return ' '
        return _node_text(node.get('c'))
    if isinstance(node, list):
        return ''.join(_node_text(item) for item in node)
    if isinstance(node, str):
        return node
    return ''


def _block_text(block: dict) -> str:
    return re.sub(r"\s+", " ", _node_text(block.get('c'))).strip()


def _section_node(level: int, title: str) -> dict:
    return {
        't': 'Section',
        'c': [
            {
                't': 'Header',
                'c': [level, [title, [], []], [{'t': 'Str', 'c': title}]],
            },
            {
                't': 'Content',
                'c': [],
            },
        ],
        'subnodes': [],
    }


def _is_toc_title(block: dict) -> bool:
    return isinstance(block, dict) and block.get('t') == 'Para' and _block_text(block) == '目录'


def _is_toc_entry(block: dict) -> bool:
    return isinstance(block, dict) and block.get('t') == 'Para' and bool(_TOC_ENTRY_RE.match(_block_text(block)))


def _drop_toc_blocks(blocks: list) -> list:
    """Remove generated table-of-contents paragraphs before section packing."""
    result = []
    in_toc = False
    for block in blocks:
        if _is_toc_title(block):
            in_toc = True
            continue
        if in_toc and _is_toc_entry(block):
            continue
        in_toc = False
        result.append(block)
    return result


def _ordered_list_heading_section(node: dict) -> Optional[dict]:
    """Convert a leading single-item ordered list like `1. Title` to a section."""
    if not isinstance(node, dict) or node.get('t') != 'OrderedList':
        return None
    attrs, items = node.get('c', [None, []])
    if not isinstance(attrs, list) or len(attrs) < 1 or attrs[0] != 1 or len(items) != 1:
        return None
    item = items[0]
    if not isinstance(item, list) or len(item) != 1:
        return None
    block = item[0]
    if not isinstance(block, dict) or block.get('t') != 'Plain':
        return None
    title = _block_text(block)
    if not title or re.search(r"[。！？；:]$", title):
        return None
    return _section_node(1, title)


def _header_section_node(node: dict) -> Optional[dict]:
    if not isinstance(node, dict) or node.get('t') != 'Header':
        return None
    return {
        't': 'Section',
        'c': [
            {
                't': 'Header',
                'c': [node['c'][0], node['c'][1], node['c'][2]],
            },
            {
                't': 'Content',
                'c': [],
            },
        ],
        'subnodes': [],
    }


def reorg_section(section: dict, it: Iterator, bIgnoreLevel:bool=False) -> tuple[dict, Iterator]:
    """Pack following nodes into a section until a sibling or parent section is reached.
    
    Args:
        section: Section node being populated.
        it: Iterator over remaining sibling nodes.
        bIgnoreLevel: Whether to ignore section-level boundary checks.
    
    Returns:
        tuple[dict, Iterator]: Updated section and iterator for the remaining nodes.
    """

    assert section.get('t') == "Section", "Expected a Section node"
    level = section['c'][0]['c'][0]

    section.setdefault('subnodes', [])

    while True:
        try:
            item = next(it)
            if not bIgnoreLevel and isinstance(item, dict) and item.get('t') == "Section":
                next_level = item['c'][0]['c'][0]
                if next_level > level:
                    # Lower-level sections are structural children, not body content.
                    item, it = reorg_section(item, it, bIgnoreLevel=False)
                    section['subnodes'].append(item)
                    continue
                elif next_level == level:  # lower levels need to be continuous
                    # If the next Section is at the same level, we can stop here at the current recursion level
                    it = iter([item] + list(it))
                    return section, it
                    # item, it = reorg_section(item, it)
                elif next_level < level:  # higher levels don't need to be continuous
                    # If the next Section is at a higher level, we can stop here at the current recursion level
                    it = iter([item] + list(it))
                    return section, it
                else:  # next_level >= level + 2:
                    raise ValueError(f"Unexpected Section level encountered: node: {item}")

            # If the item is not a Section, we can add it to the Content of the Section
            assert section['c'][1].get('t') == "Content", "Expected a Content node"
            section['c'][1]['c'].append(item)

        except StopIteration:
            break
    return section, it


In [ ]:
# | hide
def test_reorg_section_packs_children_until_sibling_and_rejects_wrong_type():
    section = {
        't': 'Section',
        'c': [
            _header(1, 'Intro'),
            {'t': 'Content', 'c': []},
        ],
    }
    items = iter([
        _para('alpha'),
        {'t': 'Section', 'c': [_header(2, 'Child'), {'t': 'Content', 'c': []}]},
        _para('beta'),
        {'t': 'Section', 'c': [_header(1, 'Sibling'), {'t': 'Content', 'c': []}]},
        _para('tail'),
    ])

    packed, remaining = reorg_section(section, items, bIgnoreLevel=False)

    test_eq(len(packed['c'][1]['c']), 1)
    test_eq(packed['c'][1]['c'][0]['t'], 'Para')
    test_eq(len(packed['subnodes']), 1)
    test_eq(packed['subnodes'][0]['t'], 'Section')
    test_eq(packed['subnodes'][0]['c'][1]['c'][0]['t'], 'Para')
    remaining_items = list(remaining)
    test_eq(remaining_items[0]['c'][0]['c'][2][0]['c'], 'Sibling')
    test_eq(remaining_items[1]['t'], 'Para')
    test_fail(lambda: reorg_section({'t': 'Para'}, iter([])), contains='Expected a Section node')


test_reorg_section_packs_children_until_sibling_and_rejects_wrong_type()

In [ ]:
# | export
def reorg_node(node, section_level: list, action: Optional[Callable] = None):
    """Reorganizes the node structure to ensure that headers are treated as sections."""
    if action:
        node = action(node)

    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                source_items = _drop_toc_blocks(value) if key == 'blocks' else value
                new_list = []
                seen_section = False
                for child in source_items:
                    if isinstance(child, (dict, list)):
                        child = reorg_node(child, section_level=section_level, action=action)

                    section_child = _header_section_node(child) if isinstance(child, dict) else None
                    if section_child is None and key == 'blocks' and not seen_section and isinstance(child, dict):
                        section_child = _ordered_list_heading_section(child)

                    if section_child is not None:
                        child = section_child
                        section_level.append(child['c'][0]['c'][0])
                        seen_section = True
                    elif isinstance(child, dict) and child.get('t') == "Section":
                        seen_section = True

                    new_list.append(child)
                it = iter(new_list)
                new_value = []
                while True:
                    try:
                        item = next(it)
                        if isinstance(item, dict) and item.get('t') == "Section":
                            item, it = reorg_section(item, it, bIgnoreLevel=False)
                        new_value.append(item)
                    except StopIteration:
                        break
                node[key] = new_value
            elif isinstance(value, dict):
                node[key] = reorg_node(value, section_level=section_level, action=action)
    elif isinstance(node, list):
        node = [
            reorg_node(child, section_level=section_level,action=action) if isinstance(child, (dict, list)) else child
            for child in node
        ]
    return node


In [ ]:
# | hide
def test_reorg_node_turns_headers_into_sections_and_keeps_siblings():
    ast = {
        'blocks': [
            _header(1, 'Intro'),
            _para('intro body'),
            _header(2, 'Details'),
            _para('details body'),
            _header(1, 'Next'),
            _para('next body'),
        ]
    }
    section_levels = []

    def marker(node):
        if isinstance(node, dict):
            node['seen'] = True
        return node

    result = reorg_node(
        ast,
        section_level=section_levels,
        action=marker,
    )

    test_eq(result['seen'], True)
    test_eq(section_levels, [1, 2, 1])
    test_eq(len(result['blocks']), 2)
    test_eq(result['blocks'][0]['t'], 'Section')
    test_eq(result['blocks'][0]['c'][0]['t'], 'Header')
    test_eq(result['blocks'][0]['c'][1]['t'], 'Content')
    test_eq(result['blocks'][0]['c'][1]['c'][0]['t'], 'Para')
    test_eq(len(result['blocks'][0]['c'][1]['c']), 1)
    test_eq(result['blocks'][0]['subnodes'][0]['c'][0]['c'][2][0]['c'], 'Details')
    test_eq(result['blocks'][0]['subnodes'][0]['c'][1]['c'][0]['t'], 'Para')
    test_eq(result['blocks'][1]['c'][0]['c'][2][0]['c'], 'Next')


test_reorg_node_turns_headers_into_sections_and_keeps_siblings()

In [ ]:
# | export
def reorg_slides(raw_json: str, raw_markdown: str, slide_splitter: str = r"(^<!--\s*Slide number:\s*\d+\s*-->$)") -> str:
    """Split slide-delimited markdown into section-shaped AST nodes.
    
    Args:
        slide_splitter: Regular expression used to detect slide separators.
    
    Returns:
        str: Reorganized AST serialized as JSON.
    """

    assert raw_json or raw_markdown, "raw_json/raw_markdown content is empty. Cannot reorganize slides."
    # Split the raw_markdown into slides
    items = re.split(slide_splitter, raw_markdown, flags=re.MULTILINE)  # type: ignore
    presentation = json.loads(raw_json)  # type: ignore
    presentation['blocks'] = []  # type: ignore
    slide_header0 = {
        't': 'Section', 
        'c': [
            {
                't': 'Header', 
                'c': [
                    1,  # Header level, can be 1, 2, 3, etc.
                    ['slide header', [], []],  # The header format list [id, formtat1, format2]
                    [{'t': 'Str', 'c': 'slide header'}],  # The header content list
                ]
            },  # Header for the slide
            {
                't': 'Content', 
                'c': []
            }
            ]  # Content of the slide
        }
    # Reorganize each slide
    slide_header = None  # Initialize slide variable
    if items[0] == "":
        # If the first slide is empty, remove it
        items = items[1:]
        slide_header = copy.deepcopy(slide_header0)
        slide_header['c'][0]['c'][1][0] = '<!-- Slide number: 0 -->'
        slide_header['c'][0]['c'][2][0]['c'] = '<!-- Slide number: 0 -->'

    for item in items:
        if re.match(r"^<!--\s*Slide number:\s*\d+\s*-->$", item):
            # If the slide is a slide splitter
            slide_header = copy.deepcopy(slide_header0)  # must be deepcopy, otherwise the slide will be modified in place
            slide_header['c'][0]['c'][1][0] = item.strip()
            slide_header['c'][0]['c'][2][0]['c'] = item.strip()
        else:
            assert slide_header, f"Slide is not defined!"
            raw_ast = pypandoc.convert_text(item.strip(), "json", "md")
            ast = json.loads(raw_ast)
            assert ast.get('blocks'), f"AST blocks are not defined in {ast}"
            it = iter(ast['blocks'])
            slide, it = reorg_section(slide_header, it, bIgnoreLevel=True)
            presentation['blocks'].append(slide)

    # Convert the list of slides back to a single JSON object
    return json.dumps(presentation, ensure_ascii=False).encode("utf-8").decode("utf-8")


In [ ]:
# | hide
def test_reorg_slides_builds_sections_from_slide_markdown():
    presentation = {'blocks': [{'t': 'Old'}]}
    slide_markdown = '\n'.join([
        '<!-- Slide number: 1 -->',
        '# Slide One',
        'First body',
        '<!-- Slide number: 2 -->',
        'Second body',
    ])

    converted = [
        json.dumps({'blocks': [_header(1, 'Slide One'), _para('First body')]}),
        json.dumps({'blocks': [_para('Second body')]}),
    ]

    with patch.object(pypandoc, 'convert_text', side_effect=converted) as convert_text:
        result = json.loads(reorg_slides(json.dumps(presentation), slide_markdown))

    test_eq(len(result['blocks']), 2)
    test_eq(result['blocks'][0]['t'], 'Section')
    test_eq(result['blocks'][0]['c'][0]['c'][2][0]['c'], '<!-- Slide number: 1 -->')
    test_eq(result['blocks'][0]['c'][1]['c'][0]['t'], 'Header')
    test_eq(result['blocks'][0]['c'][1]['c'][1]['t'], 'Para')
    test_eq(result['blocks'][1]['c'][0]['c'][2][0]['c'], '<!-- Slide number: 2 -->')
    test_eq(result['blocks'][1]['c'][1]['c'][0]['t'], 'Para')
    test_eq(convert_text.call_count, 2)
    test_fail(lambda: reorg_slides('', ''), contains='raw_json/raw_markdown content is empty')


test_reorg_slides_builds_sections_from_slide_markdown()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()